In [111]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn import metrics
import seaborn as sns
import numpy as np
import warnings

In [112]:
target_column = "health_condition"

In [113]:
df = pd.read_csv("data/train.csv")
X_test = pd.read_csv("data/test.csv")

dfs = [df, X_test]

In [114]:
df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')

for frame in dfs:
    frame.drop('id', axis=1, inplace=True)

    frame['diet_type'] = frame['diet_type'].astype('category')
    frame['gender'] = frame['gender'].astype('category')
    frame['stress_level'] = frame['stress_level'].replace({'low':0, 'medium':1, 'high':2})
    frame['sleep_quality'] = frame['sleep_quality'].replace({'poor':0, 'average':1, 'good':2})
    frame['physical_activity_level'] = frame['physical_activity_level'].replace({'sedentary':0, 'moderate':1, 'active':2})
    frame['smoking_alcohol'] = frame['smoking_alcohol'].replace({'no':0, 'occasional':1, 'yes':2})

    object_cols = ['stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol']
    for col in object_cols:
        frame[col] = frame[col].astype(np.float64)

    frame['calorie_expenditure_per_step'] = frame['calorie_expenditure'] / (frame['step_count']+1)
    frame['step_speed'] = frame['step_count'] / (frame['exercise_duration']+1)
    frame['calorie_expenditure_per_min'] = frame['calorie_expenditure'] / (frame['exercise_duration']+1)
    frame['calorie_expenditure_per_bmi'] = frame['calorie_expenditure'] / (frame['bmi']+1)

    frame["has_exercised"] = frame["exercise_duration"] > 0

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 690088 entries, 0 to 690087
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype   
---  ------                        --------------   -----   
 0   health_condition              690088 non-null  str     
 1   sleep_duration                614089 non-null  float64 
 2   heart_rate                    682255 non-null  float64 
 3   bmi                           676190 non-null  float64 
 4   calorie_expenditure           637235 non-null  float64 
 5   step_count                    676172 non-null  float64 
 6   exercise_duration             683187 non-null  float64 
 7   water_intake                  646611 non-null  float64 
 8   diet_type                     683187 non-null  category
 9   stress_level                  607277 non-null  float64 
 10  sleep_quality                 631757 non-null  float64 
 11  physical_activity_level       653467 non-null  float64 
 12  smoking_alcohol               661506 non-

In [115]:
y = df[target_column].replace({'unhealthy':0, 'at-risk':1, 'fit':2}).astype('int')
X = df.drop(target_column, axis=1)

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)
X_frame = [X_train, X_valid, X_test]

In [116]:
X_train.to_csv('data/intermediate/train_features.csv', index=False)
X_valid.to_csv('data/intermediate/valid_features.csv', index=False)
X_test.to_csv('data/intermediate/test_features.csv', index=False)

y_train.to_csv('data/intermediate/train_labels.csv', index=False)
y_valid.to_csv('data/intermediate/valid_labels.csv', index=False)

In [117]:
cat_features = ["diet_type", "gender"]

for col in cat_features:
    if not X_frame[0][col].isna().any():
        continue
    
    for frame in X_frame:
        frame[f"{col}_isna"] = frame[col].isna()

for ft in cat_features:
    for idx, frame in enumerate(X_frame):
        X_frame[idx] = pd.concat([frame, pd.get_dummies(frame[ft])], axis=1)
        X_frame[idx].drop(ft, axis=1, inplace=True)

In [118]:
X_frame[0].to_csv('data/intermediate/train_features_oheencoded.csv', index=False)
X_frame[1].to_csv('data/intermediate/valid_features_oheencoded.csv', index=False)
X_frame[2].to_csv('data/intermediate/test_features_oheencoded.csv', index=False)

In [119]:
X_frame[0].info()

<class 'pandas.DataFrame'>
Index: 552070 entries, 148924 to 121958
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   sleep_duration                491266 non-null  float64
 1   heart_rate                    545802 non-null  float64
 2   bmi                           541053 non-null  float64
 3   calorie_expenditure           509705 non-null  float64
 4   step_count                    540909 non-null  float64
 5   exercise_duration             546550 non-null  float64
 6   water_intake                  517224 non-null  float64
 7   stress_level                  485727 non-null  float64
 8   sleep_quality                 505469 non-null  float64
 9   physical_activity_level       522744 non-null  float64
 10  smoking_alcohol               529184 non-null  float64
 11  calorie_expenditure_per_step  499392 non-null  float64
 12  step_speed                    535492 non-null  float64


In [120]:
for col in X_frame[0].columns:
    if not X_frame[0][col].isna().any():
        continue
    
    for frame in X_frame:
        frame[f"{col}_isna"] = frame[col].isna()

In [121]:
missing_num_features = ["step_count", "bmi", "heart_rate", "exercise_duration", "physical_activity_level", "smoking_alcohol", "water_intake"]
missing_cat_features = ["sleep_quality", "stress_level"]

for ft in missing_num_features:
    me = X_train[ft].mean()

    for frame in X_frame:
        frame[ft] = frame[ft].fillna(me)

for ft in missing_cat_features:
    mode = X_train[ft].mode().iloc[0]

    for frame in X_frame:
        frame[ft] = frame[ft].fillna(mode)

In [122]:
def group_numerical_impute(frame, impute_col, group_col, bins=8):
    frame[f"{group_col}_binned"] = pd.cut(frame[group_col], bins=bins)
    frame[impute_col] = frame.groupby(by=[f"{group_col}_binned"])[impute_col].transform(lambda x: x.fillna(x.mean()))
    frame.drop(f"{group_col}_binned", axis=1, inplace=True)

In [123]:
for frame in X_frame:
    group_numerical_impute(frame, "calorie_expenditure", "exercise_duration")
    group_numerical_impute(frame, "sleep_duration", "stress_level")

    frame["sleep_duration"] = frame.groupby(by=[f"sleep_quality"])["sleep_duration"].transform(lambda x: x.fillna(x.mean()))

In [124]:
for frame in X_frame:
    frame['calorie_expenditure_per_step'] = frame['calorie_expenditure'] / (frame['step_count']+1)
    frame['step_speed'] = frame['step_count'] / (frame['exercise_duration']+1)
    frame['calorie_expenditure_per_min'] = frame['calorie_expenditure'] / (frame['exercise_duration']+1)
    frame['calorie_expenditure_per_bmi'] = frame['calorie_expenditure'] / (frame['bmi']+1)

In [125]:
X_frame[0].to_csv('data/intermediate/train_features_oheencoded_imputed.csv', index=False)
X_frame[1].to_csv('data/intermediate/valid_features_oheencoded_imputed.csv', index=False)
X_frame[2].to_csv('data/intermediate/test_features_oheencoded_imputed.csv', index=False)

In [126]:
X_frame[0].info()

<class 'pandas.DataFrame'>
Index: 552070 entries, 148924 to 121958
Data columns (total 39 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   sleep_duration                     552070 non-null  float64
 1   heart_rate                         552070 non-null  float64
 2   bmi                                552070 non-null  float64
 3   calorie_expenditure                552070 non-null  float64
 4   step_count                         552070 non-null  float64
 5   exercise_duration                  552070 non-null  float64
 6   water_intake                       552070 non-null  float64
 7   stress_level                       552070 non-null  float64
 8   sleep_quality                      552070 non-null  float64
 9   physical_activity_level            552070 non-null  float64
 10  smoking_alcohol                    552070 non-null  float64
 11  calorie_expenditure_per_step       552070 non-null

In [127]:
for frame in X_frame:
    frame["calorie_expenditure_per_step"] = np.log1p(frame["calorie_expenditure_per_step"])
    frame["step_speed"] = np.log1p(frame["step_speed"])
    frame["calorie_expenditure_per_min"] = np.log1p(frame["calorie_expenditure_per_min"])

In [128]:
std_columns = ["sleep_duration", "heart_rate", "bmi", "calorie_expenditure", "exercise_duration", "water_intake", "calorie_expenditure_per_step", "step_speed", "calorie_expenditure_per_min", "calorie_expenditure_per_bmi"]
scaler = StandardScaler()

X_frame[0][std_columns] = scaler.fit_transform(X_frame[0][std_columns])
X_frame[1][std_columns] = scaler.transform(X_frame[1][std_columns])
X_frame[2][std_columns] = scaler.transform(X_frame[2][std_columns])

In [129]:
scaler = MinMaxScaler()

X_frame[0][["step_count"]] = scaler.fit_transform(X_frame[0][["step_count"]])
X_frame[1][["step_count"]] = scaler.transform(X_frame[1][["step_count"]])
X_frame[2][["step_count"]] = scaler.transform(X_frame[2][["step_count"]])

In [130]:
X_frame[0].to_csv('data/intermediate/train_features_oheencoded_imputed_scaled.csv', index=False)
X_frame[1].to_csv('data/intermediate/valid_features_oheencoded_imputed_scaled.csv', index=False)
X_frame[2].to_csv('data/intermediate/test_features_oheencoded_imputed_scaled.csv', index=False)

In [131]:
X_frame[0].info()

<class 'pandas.DataFrame'>
Index: 552070 entries, 148924 to 121958
Data columns (total 39 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   sleep_duration                     552070 non-null  float64
 1   heart_rate                         552070 non-null  float64
 2   bmi                                552070 non-null  float64
 3   calorie_expenditure                552070 non-null  float64
 4   step_count                         552070 non-null  float64
 5   exercise_duration                  552070 non-null  float64
 6   water_intake                       552070 non-null  float64
 7   stress_level                       552070 non-null  float64
 8   sleep_quality                      552070 non-null  float64
 9   physical_activity_level            552070 non-null  float64
 10  smoking_alcohol                    552070 non-null  float64
 11  calorie_expenditure_per_step       552070 non-null